In [31]:
#Load ONLY what you need

In [32]:
import pandas as pd

file_path = "complaints-2026-04-06_23_56.csv"

# reading header
preview = pd.read_csv(file_path, nrows=5)

print(preview.columns.tolist())
preview.head()

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,01/05/24,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,01/05/24,Closed with non-monetary relief,Yes,NaN,8113747
1,01/28/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,Urgent : Disputed Inquiries on Credit Report -...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,937XX,NaN,Consent provided,Web,01/28/24,Closed with explanation,Yes,NaN,8235233
2,12/08/23,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,This is XXXX XXXX do not deny my complaint by ...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,PA,15001,NaN,Consent provided,Web,12/08/23,Closed with explanation,Yes,NaN,7970275
3,04/23/19,Debt collection,Payday loan debt,Attempts to collect debt not owed,Debt was result of identity theft,National credit adjusters after giving then po...,NaN,"National Credit Adjusters, LLC",TX,791XX,NaN,Consent provided,Web,04/23/19,Closed with explanation,Yes,NaN,3220066
4,12/28/23,Debt collection,Credit card debt,False statements or representation,Attempted to collect wrong amount,"XXXX 2021, amount of XXXX to collect is Not ac...",NaN,Resurgent Capital Services L.P.,KY,402XX,NaN,Consent provided,Web,12/28/23,Closed with explanation,Yes,NaN,8070648


In [ ]:
import pandas as pd

file_path = "complaints-2026-04-06_23_56.csv"

cols = [
    'Product',
    'Sub-product',
    'Issue',
    'Sub-issue',
    'Consumer complaint narrative',
    'State',
    'Date received'
]

df = pd.read_csv(
    file_path,
    usecols=cols,
    engine='python'
)

print(df.shape)
df.head()

In [ ]:
#DROP rows without text (VERY IMPORTANT)

In [ ]:
df = df.dropna(subset=['Consumer complaint narrative'])

In [ ]:
#Clean the text

In [ ]:
df['Consumer complaint narrative'] = df['Consumer complaint narrative'].str.strip()
df = df[df['Consumer complaint narrative'].str.len() > 20]

In [ ]:
#BASIC CLEANING

In [ ]:
df = df.drop_duplicates(subset=['Consumer complaint narrative'])

In [ ]:
#FILTER TO RELEVANT PRODUCTS

In [ ]:
relevant_products = [
    'Credit reporting or other personal consumer reports',
    'Debt collection',
    'Credit card',
    'Checking or savings account',
    'Money transfer, virtual currency, or money service',
    'Mortgage',
    'Student loan'
]

df = df[df['Product'].isin(relevant_products)]

In [ ]:
#FILTER TO FRAUD / RISK ISSUES

In [ ]:
relevant_issues = [
    'Fraud or scam',
    'Unauthorized transactions or other transaction problem',
    'Identity theft protection or other monitoring services',
    'Incorrect information on your report',
    'Improper use of your report',
    'Attempts to collect debt not owed',
    'False statements or representation',
    'Communication tactics',
    'Problem with a purchase shown on your statement',
    'Trouble using your card',
    'Problem with fraud alerts or security freezes'
]

df = df[df['Issue'].isin(relevant_issues)]

In [ ]:
#SIMPLIFY LABELS (VERY IMPORTANT)

In [ ]:
label_map = {
    'Fraud or scam': 'fraud',
    'Unauthorized transactions or other transaction problem': 'fraud',
    'Identity theft protection or other monitoring services': 'identity_theft',
    'Incorrect information on your report': 'credit_issue',
    'Improper use of your report': 'credit_issue',
    'Attempts to collect debt not owed': 'debt_fraud',
    'False statements or representation': 'scam',
    'Communication tactics': 'scam',
    'Problem with a purchase shown on your statement': 'transaction_issue',
    'Trouble using your card': 'transaction_issue',
    'Problem with fraud alerts or security freezes': 'security_issue'
}

df['label'] = df['Issue'].map(label_map)

# Drop unmapped rows (IMPORTANT)
df = df.dropna(subset=['label'])

In [25]:
#DROP UNUSED COLUMNS

In [26]:
df = df[['Consumer complaint narrative', 'label', 'Product', 'State', 'Issue', 'Sub-issue', 'Date received']]

In [27]:
#CHECK RESULT

In [28]:
print(df.shape)
print(df['label'].value_counts())
print(df['Issue'].value_counts())

(708603, 7)
label
credit_issue         506801
debt_fraud           103618
scam                  46105
fraud                 24958
transaction_issue     22238
security_issue         4411
identity_theft          472
Name: count, dtype: int64
Issue
Incorrect information on your report                      321603
Improper use of your report                               185198
Attempts to collect debt not owed                         103618
False statements or representation                         28846
Problem with a purchase shown on your statement            19032
Fraud or scam                                              18470
Communication tactics                                      17259
Unauthorized transactions or other transaction problem      6488
Problem with fraud alerts or security freezes               4411
Trouble using your card                                     3206
Identity theft protection or other monitoring services       472
Name: count, dtype: int64


In [29]:
#SAVE CLEAN DATASET

In [30]:
df.to_parquet("cfpb_clean.parquet")